Подключаем Google Disk для сохранения файлов и доступа к файлам модели:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Скачиваем нужные зависимости:

In [ ]:
!pip install datasets pyannote.metrics pyannote.audio huggingface_hub

Выполняем логин в Hugging Face Hub:

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

try:
    print("Logging in HuggingFace...")
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Token has downloaded from Colab's secrets!")
    login(token=hf_token)
    print("Login successfully!")
except Exception as e:
    print(f"Error: {e}")
    hf_token = None

Собираем пайплайн из дообученной модели сегментации и подобранных гиперпараметров:

In [ ]:
import torch
from pyannote.audio import Pipeline, Model
from pyannote.audio.pipelines import SpeakerDiarization
from pyannote.audio.pipelines.clustering import AgglomerativeClustering
from pyannote.audio.utils.powerset import Powerset

pretrained_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1"
)

# Прописать путь к чекпоинту с дообученной моделью сегментации
CHECKPOINT_PATH = "/content/drive/MyDrive/pyannote_finetuning/ami_segmentation_v1/fixed_version/checkpoints/best-epoch=09-step=5329.ckpt"
segmentation_model = Model.from_pretrained(
    CHECKPOINT_PATH
)

custom_pipeline = SpeakerDiarization(
    segmentation=segmentation_model,
    embedding=pretrained_pipeline.embedding,
    embedding_exclude_overlap=pretrained_pipeline.embedding_exclude_overlap,
    clustering="AgglomerativeClustering"
)

Создаем 4 конфигурации ошибки DER: strict, collar, no overlaps, clean

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate, DiarizationPurity, DiarizationCoverage
from pyannote.metrics.detection import DetectionErrorRate

configs = {
    'strict': {'collar': 0.0, 'skip_overlap': False},
    'collar': {'collar': 0.25, 'skip_overlap': False},
    'no_ovl': {'collar': 0.0, 'skip_overlap': True},
    'clean':  {'collar': 0.25, 'skip_overlap': True}
}

metrics_vault = {}
for c_name, params in configs.items():
    metrics_vault[c_name] = {
        'der': DiarizationErrorRate(**params),
        'purity': DiarizationPurity(**params),
        'coverage': DiarizationCoverage(**params),
        'det': DetectionErrorRate(collar=params['collar']) # VAD is independent of overlap
    }

Ставим гиперпараметры, полученные после оптимизации с помощью фрэймворка Optuna:

In [ ]:
custom_params = {
        "segmentation": {
            "min_duration_off": 0.3537883320126253,
        },
        "clustering": {
            "method": 'centroid',
            "min_cluster_size": 19,
            "threshold": 0.7172270484758725,
        }
    }

custom_pipeline.instantiate(custom_params)
print(custom_pipeline.parameters(instantiated=True))

Загружаем тестовую выборку из датасета:

In [ ]:
from pyannote.database import registry, get_protocol, FileFinder
from pyannote.audio import Pipeline
import torch
import os
from pyannote.core import Annotation, Segment
import pandas as pd
from datetime import datetime

# Нужно поставить путь до конфига database.yml
DATABASE_CONFIG_FILE = "/content/drive/MyDrive/AMI-diarization-setup/pyannote/database.yml"
PROTOCOL = "AMI.SpeakerDiarization.word_and_vocalsounds"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

registry.load_database(DATABASE_CONFIG_FILE)
protocol = registry.get_protocol(PROTOCOL, preprocessors={"audio": FileFinder()})
test_set = protocol.test()

Выполняем тестирование пайплайна Pyannote с дообученной моделью сегментации и подобранными гиперпараметрами:

In [ ]:
results = []

for i, test_item in enumerate(test_set):
    print(f"Processing {i}: {test_item['uri']}")
    audio_path = test_item["audio"]
    reference = test_item["annotation"]
    custom_pipeline.to(torch.device(DEVICE))
    hypothesis = custom_pipeline(audio_path)
    if hasattr(hypothesis, "speaker_diarization"):
        hypothesis = hypothesis.speaker_diarization

    res = {'file': test_item['uri']}
    for c_name, m_group in metrics_vault.items():
        der = m_group['der'](reference, hypothesis)
        purity = m_group['purity'](reference, hypothesis)
        coverage = m_group['coverage'](reference, hypothesis)
        deter = m_group['det'](reference, hypothesis)

        components = m_group['der'].compute_components(reference, hypothesis)
        fa = components['false alarm']
        miss = components['missed detection']
        conf = components['confusion']
        total = components['total']

        res[f'DER_{c_name}'] = der
        res[f'Purity_{c_name}'] = purity
        res[f'Coverage_{c_name}'] = coverage
        res[f'DetER_{c_name}'] = deter
        res[f'FA_{c_name}'] = fa
        res[f'Miss_{c_name}'] = miss
        res[f'Conf_{c_name}'] = conf
        res[f'Total_Speech_{c_name}'] = total

        print(f"- {c_name} - DER={der:.2%} FA={fa:.2%} Miss={miss:.2%} Conf={conf:.2%}")

    results.append(res)
    print("==================================================")

Сохранение результатов тестирования в csv файл:

In [ ]:
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
df = pd.DataFrame(results)
os.makedirs("results", exists_ok=True)
# При необходимости можно поменять путь
df.to_csv(f"results/custom_pyannote_metrics_{current_time}.csv", index=False)

Визуализация графиков с результатами от пайплайна с дообученной моделью сегментации:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Заменить путь на правильный
df = pd.read_csv("/content/drive/MyDrive/Testing_pro_diarization_pipeline/custom_pyannote_metrics_20260508_204531.csv")

res = df.mean(numeric_only=True).to_dict()

visualize_configs = ["strict", "collar", "no_ovl", "clean"]
for c_name in visualize_configs:
    der = res[f'DER_{c_name}']
    total = res[f'Total_Speech_{c_name}']
    fa = (res[f'FA_{c_name}'] / total) * 100
    miss = (res[f'Miss_{c_name}'] / total) * 100
    conf = (res[f'Conf_{c_name}'] / total) * 100
    purity = res[f'Purity_{c_name}'] * 100
    coverage = res[f'Coverage_{c_name}'] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle(f'Configuration: {c_name.upper()}, DER: {round(der * 100, 3)}%', fontsize=18, fontweight='bold', y=1.02)

    # Graph 1. Components of DER
    labels = ['Missed', 'False Alarm', 'Confusion']
    errors = [miss, fa, conf]
    colors = ['#FF6B6B', '#4D96FF', '#6BCB77']

    bars1 = ax1.bar(labels, errors, color=colors, edgecolor='black', alpha=0.8)
    max_err = max(errors)
    upper_limit = max_err * 1.15 if max_err > 0 else 10

    ax1.set_ylim(0, upper_limit)
    ax1.set_title(f'DER Components', fontsize=14, pad=20)
    ax1.set_ylabel('Error Rate (%)')
    ax1.grid(axis='y', linestyle='--', alpha=0.6)

    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2., height + (upper_limit * 0.01),
                f'{height:.2f}%', ha='center', va='bottom',
                fontweight='bold', fontsize=11, color='black')

    # Graph 2. Purity & Coverage Quality
    q_labels = ['Purity', 'Coverage']
    q_values = [purity, coverage]
    q_colors = ['#FFD93D', '#A084CA']

    bars2 = ax2.bar(q_labels, q_values, color=q_colors, edgecolor='black', width=0.5)

    ax2.set_ylim(0, 105)
    ax2.set_title('Quality Metrics (Higher is Better)', fontsize=14, pad=20)
    ax2.set_ylabel('Score (%)')
    ax2.grid(axis='y', linestyle='--', alpha=0.3)

    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2., height + 1,
                f'{height:.1f}%', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

    plt.tight_layout()
    plt.savefig(f'der_components_{c_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print()

Загрузка тестовой выборки для тестирования базового пайплайна Pyannote:

In [ ]:
# Нужно поставить путь до конфига database.yml
DATABASE_CONFIG_FILE = "/content/drive/MyDrive/AMI-diarization-setup/pyannote/database.yml"
PROTOCOL = "AMI.SpeakerDiarization.word_and_vocalsounds"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

registry.load_database(DATABASE_CONFIG_FILE)
protocol = registry.get_protocol(PROTOCOL, preprocessors={"audio": FileFinder()})
test_set = protocol.test()

Тестирование базового пайплайна Pyannote:

In [ ]:
results = []

for i, test_item in enumerate(test_set):
    print(f"Processing {i}: {test_item['uri']}")
    audio_path = test_item["audio"]
    reference = test_item["annotation"]
    pretrained_pipeline.to(torch.device(DEVICE))
    hypothesis = pretrained_pipeline(audio_path)
    if hasattr(hypothesis, "speaker_diarization"):
        hypothesis = hypothesis.speaker_diarization

    res = {'file': test_item['uri']}
    for c_name, m_group in metrics_vault.items():
        der = m_group['der'](reference, hypothesis)
        purity = m_group['purity'](reference, hypothesis)
        coverage = m_group['coverage'](reference, hypothesis)
        deter = m_group['det'](reference, hypothesis)

        components = m_group['der'].compute_components(reference, hypothesis)
        fa = components['false alarm']
        miss = components['missed detection']
        conf = components['confusion']
        total = components['total']

        res[f'DER_{c_name}'] = der
        res[f'Purity_{c_name}'] = purity
        res[f'Coverage_{c_name}'] = coverage
        res[f'DetER_{c_name}'] = deter
        res[f'FA_{c_name}'] = fa
        res[f'Miss_{c_name}'] = miss
        res[f'Conf_{c_name}'] = conf
        res[f'Total_Speech_{c_name}'] = total

        print(f"- {c_name} - DER={der:.2%} FA={fa:.2%} Miss={miss:.2%} Conf={conf:.2%}")

    results.append(res)
    print("==================================================")

Сохраняем результаты от базового пайплайна Pyannote:

In [ ]:
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
df = pd.DataFrame(results)
os.makedirs("results", exists_ok=True)
# Поменять путь, если нужно
df.to_csv(f"results/base_pipeline_pyannote_metrics_{current_time}.csv", index=False)

Визуализация результатов:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Заменить путь на нужный. Это путь до итогового файла csv после тестирования
df = pd.read_csv("/content/drive/MyDrive/Testing_pro_diarization_pipeline/base_pipeline_pyannote_metrics_20260508_210527.csv")

res = df.mean(numeric_only=True).to_dict()
print(res)

visualize_configs = ["strict", "collar", "no_ovl", "clean"]
for c_name in visualize_configs:
    der = res[f'DER_{c_name}']
    total = res[f'Total_Speech_{c_name}']
    fa = (res[f'FA_{c_name}'] / total) * 100
    miss = (res[f'Miss_{c_name}'] / total) * 100
    conf = (res[f'Conf_{c_name}'] / total) * 100
    purity = res[f'Purity_{c_name}'] * 100
    coverage = res[f'Coverage_{c_name}'] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle(f'Configuration: {c_name.upper()}, DER: {round(der * 100, 3)}%', fontsize=18, fontweight='bold', y=1.02)

    # Graph 1. Components of DER
    labels = ['Missed', 'False Alarm', 'Confusion']
    errors = [miss, fa, conf]
    colors = ['#FF6B6B', '#4D96FF', '#6BCB77']

    bars1 = ax1.bar(labels, errors, color=colors, edgecolor='black', alpha=0.8)
    max_err = max(errors)
    upper_limit = max_err * 1.15 if max_err > 0 else 10

    ax1.set_ylim(0, upper_limit)
    ax1.set_title(f'DER Components', fontsize=14, pad=20)
    ax1.set_ylabel('Error Rate (%)')
    ax1.grid(axis='y', linestyle='--', alpha=0.6)

    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2., height + (upper_limit * 0.01),
                f'{height:.2f}%', ha='center', va='bottom',
                fontweight='bold', fontsize=11, color='black')

    # Graph 2. Purity & Coverage Quality
    q_labels = ['Purity', 'Coverage']
    q_values = [purity, coverage]
    q_colors = ['#FFD93D', '#A084CA']

    bars2 = ax2.bar(q_labels, q_values, color=q_colors, edgecolor='black', width=0.5)

    ax2.set_ylim(0, 105)
    ax2.set_title('Quality Metrics (Higher is Better)', fontsize=14, pad=20)
    ax2.set_ylabel('Score (%)')
    ax2.grid(axis='y', linestyle='--', alpha=0.3)

    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2., height + 1,
                f'{height:.1f}%', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

    plt.tight_layout()
    plt.savefig(f'der_components_{c_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print()